## Phase 6.6.1 — Number of Clinical Trials


In [ ]:
Phase_6_6_Clinical_Features.ipynb

6.6.1 Clinical trial count (RepoDB)
6.6.2 FDA approval status
6.6.3 Side effects


In [104]:
import os

base_dir = r"C:\Users\deep8\breast_cancer_project_folder"

# list all files in project folder
files = sorted(os.listdir(base_dir))
files


['.ipynb_checkpoints',
 'CRISPR_gene_effect.csv',
 'Census_allWed Jan 28 09_32_24 2026.csv',
 'DEGs_HER2_vs_Normal_FDR0.05_log2FC1.csv',
 'DEGs_LumA_vs_Normal_FDR0.05_log2FC1.csv',
 'DEGs_LumB_vs_Normal_FDR0.05_log2FC1.csv',
 'DEGs_TNBC_vs_Normal_FDR0.05_log2FC1.csv',
 'Drug_CRISPR_Essentiality_Global.csv',
 'Drug_CRISPR_Essentiality_HER2.csv',
 'Drug_CRISPR_Essentiality_Luminal.csv',
 'Drug_CRISPR_Essentiality_TNBC.csv',
 'Drug_HER2_MinShortestPath.csv',
 'Drug_HER2_Propagation_Overlap.csv',
 'Drug_LINCS_Connectivity_HER2.csv',
 'Drug_LINCS_Connectivity_LumA.csv',
 'Drug_LumA_Propagation_Overlap.csv',
 'Drug_LumA_RW_Proximity.csv',
 'Drug_LumA_ShortestPath.csv',
 'Drug_LumB_Propagation_Overlap.csv',
 'Drug_LumB_RW_Proximity.csv',
 'Drug_LumB_ShortestPath.csv',
 'Drug_Pathway_Enrichment_Similarity_HER2.csv',
 'Drug_Pathway_Enrichment_Similarity_LumA.csv',
 'Drug_Pathway_Enrichment_Similarity_LumB.csv',
 'Drug_Pathway_Enrichment_Similarity_TNBC.csv',
 'Drug_Pathway_Features_HER2.csv',
 

In [3]:
import pandas as pd

# load drug–target mapping (PPI-filtered, Phase 6.1 output)
drug_targets = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Targets_PPI_Filtered.csv"
)

# inspect columns
drug_targets.columns


Index(['target_chembl_id', 'targets', 'targets_ppi'], dtype='object')

In [5]:
import pandas as pd

# load mapping again (safe)
drug_targets = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Targets_PPI_Filtered.csv"
)

# extract unique drugs
drug_list = (
    drug_targets[["target_chembl_id"]]
    .drop_duplicates()
    .rename(columns={"target_chembl_id": "chembl_id"})
    .reset_index(drop=True)
)

drug_list.shape, drug_list.head()


((522, 1),
     chembl_id
 0  CHEMBL1778
 1  CHEMBL1782
 2  CHEMBL1783
 3  CHEMBL1785
 4  CHEMBL1786)

In [7]:
import pandas as pd

chembl_reps = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\chembl_36_chemreps.txt",
    sep="\t"
)

chembl_reps.columns


Index(['chembl_id', 'canonical_smiles', 'standard_inchi',
       'standard_inchi_key'],
      dtype='object')

In [9]:
import json
import pandas as pd

# load ChEMBL mechanism file
with open(r"C:\Users\deep8\breast_cancer_project_folder\chembl_mechanism.json") as f:
    mech = json.load(f)

mech_df = pd.json_normalize(mech)

# inspect columns
mech_df.columns


Index(['mechanisms', 'page_meta.limit', 'page_meta.next', 'page_meta.offset',
       'page_meta.previous', 'page_meta.total_count'],
      dtype='object')

In [124]:
mech_df.columns

Index(['action_type', 'binding_site_comment', 'direct_interaction',
       'disease_efficacy', 'max_phase', 'mec_id', 'mechanism_comment',
       'mechanism_of_action', 'mechanism_refs', 'molecular_mechanism',
       'molecule_chembl_id', 'parent_molecule_chembl_id', 'record_id',
       'selectivity_comment', 'site_id', 'target_chembl_id',
       'variant_sequence', 'variant_sequence.accession',
       'variant_sequence.isoform', 'variant_sequence.mutation',
       'variant_sequence.organism', 'variant_sequence.sequence',
       'variant_sequence.tax_id', 'variant_sequence.version'],
      dtype='object')

In [19]:
# extract regulatory info from ChEMBL mechanism table
regulatory_df = (
    mech_df[["molecule_chembl_id", "max_phase"]]
    .drop_duplicates()
    .rename(columns={"molecule_chembl_id": "chembl_id"})
)

# merge with master drug list
drug_list = drug_list.merge(
    regulatory_df,
    on="chembl_id",
    how="left"
)

# fill missing as preclinical
drug_list["max_phase"] = drug_list["max_phase"].fillna(0).astype(int)

# FDA approval flag
drug_list["fda_approved"] = (drug_list["max_phase"] >= 4).astype(int)

drug_list.head(), drug_list["fda_approved"].value_counts()


KeyError: "None of [Index(['molecule_chembl_id', 'max_phase'], dtype='object')] are in the [columns]"

In [21]:
import pandas as pd

# load SIDER drug–side effect mapping
# (assumes drug names / IDs mapped earlier)
sider = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathways.csv"
)

sider.columns


Index(['drug', 'reactome_id', 'pathway_name'], dtype='object')

In [23]:
import pandas as pd

# load drug–pathway associations
drug_pathways = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Pathways.csv"
)

# count number of pathways per drug
pathway_burden = (
    drug_pathways.groupby("drug")
    .size()
    .reset_index(name="num_pathways_affected")
)

pathway_burden.head(), pathway_burden.shape


(         drug  num_pathways_affected
 0  CHEMBL1778                     16
 1  CHEMBL1782                      7
 2  CHEMBL1783                     29
 3  CHEMBL1785                     10
 4  CHEMBL1786                      3,
 (512, 2))

In [17]:
# rename pathway_burden key to match drug_list
pathway_burden_fixed = pathway_burden.rename(
    columns={"drug": "chembl_id"}
)

# merge correctly
drug_list = drug_list.merge(
    pathway_burden_fixed,
    on="chembl_id",
    how="left"
)

# fill missing values
drug_list["num_pathways_affected"] = (
    drug_list["num_pathways_affected"]
    .fillna(0)
    .astype(int)
)

drug_list.head(), drug_list.shape


NameError: name 'pathway_burden' is not defined

In [15]:
drug_list.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Clinical_Regulatory_Features.csv",
    index=False
)

drug_list.shape


(522, 1)